In [112]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns


In [113]:
df=pd.read_csv("CustomerChurn.csv")

In [114]:
df.sample(10)

,LoyaltyID,Customer ID,Senior Citizen,Partner,Dependents,Tenure,Phone Service,Multiple Lines,Internet Service,Online Security,...,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn
1254,111788,9220-ZNKJI,Yes,Yes,No,55,Yes,No,Fiber optic,No,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),88.80,4805.3,No
1392,688557,1934-MKPXS,No,Yes,Yes,33,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,No,Credit card (automatic),20.10,620.55,No
5998,178821,0442-TDYUO,No,Yes,No,48,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Mailed check,20.05,1036,No
3242,792514,3717-FDJFU,No,No,Yes,5,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.45,106.9,No
2462,813185,2585-KTFRE,No,No,Yes,1,Yes,No,DSL,Yes,...,No,Yes,No,Yes,Month-to-month,Yes,Bank transfer (automatic),70.45,70.45,No
1140,461012,9553-DLCLU,No,No,Yes,13,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),88.95,1161.75,No
3771,203033,5960-WPXQM,No,No,No,1,Yes,No,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Electronic check,79.05,79.05,Yes
6851,935543,8465-SBRXP,No,Yes,Yes,38,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),94.90,3616.25,No
2522,253656,0661-KBKPA,No,Yes,Yes,53,Yes,Yes,DSL,No,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,78.75,3942.45,No
687,271641,0067-DKWBL,Yes,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,49.25,91.1,Yes


In [116]:
from ydata_profiling import ProfileReport
prof=ProfileReport(df)
prof.to_file(output_file="output.html")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 21/21 [00:00<00:00, 1764.54it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [117]:
df.columns = df.columns.str.strip()

In [118]:

df.drop(columns=["LoyaltyID","Customer ID","Phone Service"],inplace=True)
df.head()


,Senior Citizen,Partner,Dependents,Tenure,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn
0,No,Yes,No,1,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,No,No,No,34,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,No,No,No,2,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,No,No,No,45,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,No,No,No,2,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [119]:
df.shape


(7043, 18)

In [134]:
df["Total Charges"] = pd.to_numeric(
    df["Total Charges"].replace(r"^\s*$", np.nan, regex=True),
    errors="coerce"
)
df.replace("No Phone Service","No",inplace=True)
df.replace("No Internet Service","No",inplace=True)



In [151]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df.drop(columns=["Churn"]),df["Churn"],test_size=0.2,random_state=42)

In [152]:
X_train.sample(5)

,Senior Citizen,Partner,Dependents,Tenure,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges
809,No,No,No,1,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.55,69.55
4197,No,No,Yes,22,Yes,Fiber optic,No,No,No,No,No,Yes,Month-to-month,Yes,Electronic check,84.75,1816.75
2818,No,Yes,Yes,29,No,Fiber optic,Yes,No,No,No,Yes,Yes,Month-to-month,No,Credit card (automatic),94.65,2649.15
3958,No,Yes,No,16,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),73.85,1284.20
6183,No,Yes,Yes,44,No,DSL,No,No,Yes,Yes,Yes,Yes,One year,Yes,Electronic check,54.30,2317.10


In [153]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder, LabelEncoder
from sklearn.preprocessing import StandardScaler 
from sklearn.impute import SimpleImputer

In [154]:
transformer = ColumnTransformer(transformers=[
    ("impute",SimpleImputer(strategy="median"),["Total Charges"]),
    ("tnf1", StandardScaler(), ["Monthly Charges", "Total Charges", "Tenure"]),

    ("tnf2",
     OrdinalEncoder(categories=[["No", "Yes"]] * 11),
     ["Senior Citizen", "Partner", "Dependents", "Multiple Lines",
      "Online Security", "Online Backup", "Device Protection",
      "Tech Support", "Streaming TV", "Streaming Movies","Paperless Billing"]),

    ("tnf3",
     OneHotEncoder(sparse_output=False, drop=["No"]),
     ["Internet Service"]),

    ("tnf4",
     OrdinalEncoder(categories=[["Month-to-month", "One year", "Two year"]]),
     ["Contract"]),

    ("tnf5",
     OneHotEncoder(sparse_output=False, drop="first"),
     ["Payment Method"])
    

], remainder="passthrough")

In [156]:
le=LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
y_train

array([0, 0, 1, ..., 0, 1, 0], shape=(5634,))

In [186]:
from sklearn.linear_model import LogisticRegression
lr=LogisticRegression(max_iter=1000)
type(lr)

sklearn.linear_model._logistic.LogisticRegression

In [187]:
from sklearn.pipeline import Pipeline,make_pipeline
pipe=make_pipeline(transformer,lr)

In [190]:
pipe.fit(X_train,y_train)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,steps,"[('columntransformer', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('impute', ...), ('tnf1', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [204]:
y_pred1 = (pipe.predict_proba(X_test)[:, 1] >= 0.3).astype(int)
y_pred1

array([1, 0, 0, ..., 0, 0, 1], shape=(1409,))

In [207]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,precision_score,recall_score

print("Accuracy= ", accuracy_score(y_test, y_pred1))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred1))

print("\nClassification report:")
print(classification_report(y_test, y_pred1))
print("Recall =",recall_score(y_test,y_pred1))
print("Precision =",precision_score(y_test,y_pred1))



Accuracy=  0.7750177430801988

Confusion matrix:
[[793 243]
 [ 74 299]]

Classification report:
              precision    recall  f1-score   support

           0       0.91      0.77      0.83      1036
           1       0.55      0.80      0.65       373

    accuracy                           0.78      1409
   macro avg       0.73      0.78      0.74      1409
weighted avg       0.82      0.78      0.79      1409

Recall = 0.8016085790884718
Precision = 0.551660516605166


In [206]:
'''Recall is my first priority here as if I miss a churn and the customer leaves it is more costly than 
predicting customer might leave and provide them some discount and they don't actually leave that is the
case of precision'''

"Recall is my first priority here as if I miss a churn and the customer leaves it is more costly than \npredicting customer might leave and provide them some discount and they don't actually leave that is the\ncase of precision"